# Практическое занятие № 22 — Предзащита ML-проекта и анализ готовности к внедрению
**ТюмГУ · Технологии искусственного интеллекта — 1 · 90 минут**  
Автор учебных материалов: В. А. Мазур, доцент ШКН ТюмГУ.

**Результат:** провести технический аудит пакета из практики № 21, доказать воспроизводимость инференса, проверить границы достоверности результатов, составить `model_card.md`, `deployment_readiness.csv` и план исправлений, подготовить 5-минутную защиту.

**Для запуска:** разместите рядом с ноутбуком `results_21_bundle.zip`, созданный в практике № 21, **или** приложенный `results_21_bundle_demo.zip` (из набора преподавателя). Второй вариант содержит только искусственные демонстрационные данные; он не доказывает качество студентской модели. В Google Colab загрузите архив в каталог `/content` через панель «Файлы».

## 1. Загрузка артефактов и журнал источника (0–12 мин)

Проверка включает происхождение набора, запретные признаки, размеры разделения, правила выбора checkpoint и порога, готовность схемы входов и наличие необходимых файлов. Не загружайте `.joblib`/`.pt` из недоверенных источников.

In [ ]:
from pathlib import Path
import zipfile,json,time,platform,sys
import numpy as np,pandas as pd,matplotlib.pyplot as plt
import torch,joblib,sklearn
from torch import nn
from sklearn.metrics import average_precision_score, confusion_matrix, brier_score_loss, precision_score, recall_score, f1_score
from IPython.display import display
R=Path('results_21')
if not R.is_dir():
    arc=next((Path(a) for a in ['results_21_bundle.zip','results_21_bundle_demo.zip'] if Path(a).exists()),None)
    if arc is None:raise FileNotFoundError('Поместите results_21_bundle.zip из практики 21 или results_21_bundle_demo.zip рядом с блокнотом.')
    with zipfile.ZipFile(arc) as z:
        names=z.namelist()
        if not all(name.startswith('results_21/') and '..' not in Path(name).parts for name in names):
            raise ValueError('Недопустимая структура архива.')
        z.extractall('.') # архив получен только от преподавателя или собственный, не чужой
print('Каталог:',R.resolve())
manifest=json.loads((R/'manifest.json').read_text(encoding='utf-8'))
config=json.loads((R/'model_config.json').read_text(encoding='utf-8'))
print('Источник:',manifest['origin'])
print('Доверенный независимый test?',manifest['independent_test_claim'])
print('Выбранная модель / порог:',manifest['selected_model'],config['threshold'])
if 'ДЕМОНСТРАЦ' in manifest['origin'].upper() or 'ИСКУССТВЕН' in manifest['origin'].upper():
    print('ВАЖНО: демонстрационные синтетические данные. Выводы нельзя переносить на реальную логистику.')

## 2. Аудит инвариантов данных (12–23 мин)

Проверьте взаимную непересекаемость `order_id`, временной порядок, допустимость признаков и долю положительного класса. Совпадение размера набора не доказывает корректность: нужны проверки уникальности ключа, отсутствия future leakage и подтверждение словаря полей.

In [ ]:
required=['manifest.json','model_config.json','mlp_state.pt','preprocessor.joblib','split_membership.csv',
 'experiments.csv','selected_history.csv','validation_predictions.csv','validation_model_comparison.csv','inference_samples.csv','README.md']
report=[{'check':name,'status':'PASS' if (R/name).exists() else 'FAIL'} for name in required]
assert all(row['status']=='PASS' for row in report),'Неполный пакет. Требуется исправить практику № 21.'
splits=pd.read_csv(R/'split_membership.csv',parse_dates=['created_at'])
assert splits.order_id.notna().all() and not splits.order_id.duplicated().any()
sets={part:set(splits.loc[splits.part==part,'order_id']) for part in ['train','validation','test']}
assert all(sets[a].isdisjoint(sets[b]) for a,b in [('train','validation'),('train','test'),('validation','test')])
assert splits.loc[splits.part=='train','created_at'].max()<=splits.loc[splits.part=='validation','created_at'].min()
assert splits.loc[splits.part=='validation','created_at'].max()<=splits.loc[splits.part=='test','created_at'].min()
assert not (set(manifest['feature_columns']) & set(manifest['forbidden_predictors']))
assert len(splits)==manifest['work_rows']
counts=splits.groupby('part')['late'].agg(['count','sum','mean'])
display(counts)
print('Артефакты и инварианты пройдены. Но качество исходных данных и правовой статус требуют ручной проверки.')

## 3. Тест восстановления модели и схемы признаков (23–40 мин)

Критически важно восстановить не только веса, но и модель с той же архитектурой, препроцессор и **порядок признаков**. Выполните инференс на заранее отложенных образцах validation и сравните числовые вероятности с сохранёнными; это проверка **воспроизводимости**, не новый тест качества.

Эксперимент с неизвестной категорией должен **не падать**, но не доказывает, что новый регион безопасен для эксплуатации.

In [ ]:
class DeliveryMLP(nn.Module):
    def __init__(self,d,width=16,dropout=0.):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d,width),nn.ReLU(),nn.Dropout(dropout),
          nn.Linear(width,width),nn.ReLU(),nn.Dropout(dropout),nn.Linear(width,1))
    def forward(self,x):return self.net(x).squeeze(1)
# Безопасно только для СВОЕГО проверенного bundle: joblib использует pickle и исполняет произвольный код.
prep=joblib.load(R/'preprocessor.joblib')
model=DeliveryMLP(config['n_inputs'],config['width'],config['dropout'])
weights=torch.load(R/'mlp_state.pt',map_location='cpu',weights_only=True)
model.load_state_dict(weights);model.eval()
samples=pd.read_csv(R/'inference_samples.csv')
FEATURES=manifest['feature_columns']
assert all(col in samples.columns for col in FEATURES)
for col in manifest['numeric_columns']:samples[col]=pd.to_numeric(samples[col],errors='coerce')
X=prep.transform(samples[FEATURES]);X=np.asarray(X,dtype=np.float32)
assert X.shape[1]==config['n_inputs']
with torch.no_grad():probs=torch.sigmoid(model(torch.from_numpy(X))).numpy()
maxerr=float(np.max(abs(probs-samples.p_expected.to_numpy())))
assert maxerr<1e-5, f'Невозможно подтвердить восстановление вероятностей: {maxerr}'
print('Максимальное отклонение вероятности:',maxerr,'(проверка пройдена)')
# Демонстрация новой категории, не автоматическая гарантия качества
novel=samples[FEATURES].iloc[[0]].copy()
novel[manifest['categorical_columns'][0]]='UNKNOWN_CATEGORY_2026'
Xt=np.asarray(prep.transform(novel),dtype=np.float32)
with torch.no_grad(): print('Прогноз при новой категории:',float(torch.sigmoid(model(torch.from_numpy(Xt)))[0]))
print('ОГРАНИЧЕНИЕ: значения вне обученного словаря нуждаются в мониторинге и проверке экспертами.')

## 4. Проверка метрик, бизнес-правила и нагрузки (40–56 мин)

Проверяйте результаты на **validation**, а не подбирайте новую архитектуру по test. При стоимости `60·FP + 300·FN` рассчитайте фактическую частоту тревог; если пропускная способность диспетчера ограничена, сформируйте отдельный сценарий и согласуйте его с предметным экспертом.

$$C(\tau)=60FP(\tau)+300FN(\tau),\quad \text{cost}_{100}=100\,C/N$$

Независимое качество на будущем периоде остаётся **неизвестным**, если тест ранее использовался при настройке учебного проекта.

In [ ]:
v=pd.read_csv(R/'validation_predictions.csv')
tau=float(config['threshold']); pred=(v.p_mlp>=tau).astype(int)
tn,fp,fn,tp=confusion_matrix(v.late,pred,labels=[0,1]).ravel()
metrics={'AP':average_precision_score(v.late,v.p_mlp),'F1':f1_score(v.late,pred,zero_division=0),
 'Recall':recall_score(v.late,pred,zero_division=0),'Precision':precision_score(v.late,pred,zero_division=0),
 'Brier':brier_score_loss(v.late,v.p_mlp),'FP':int(fp),'FN':int(fn),'alerts':int(pred.sum()),
 'alert_rate':float(pred.mean()),'cost_total_rub':int(60*fp+300*fn),'cost_100_rub':float(100*(60*fp+300*fn)/len(v))}
display(pd.DataFrame([metrics]).T.rename(columns={0:'validation'}))
# Условная пропускная способность: не более 20 тревог на 100 заказов.
capacity=20
excess=max(0,metrics['alert_rate']*100-capacity)
print(f'Сценарий вместимости: {capacity} сигналов/100 заказов; превышение {excess:.2f} сигналов/100.')
print('Это не эксплуатационный SLA: неизвестны часы пиков, время реакции и эффект вмешательства.')
fig,ax=plt.subplots(figsize=(6.4,3.6))
ax.bar(['TP','FP','FN','TN'],[tp,fp,fn,tn]);ax.set_ylabel('Заказов (validation)');ax.set_title('Распределение исходов при выбранном пороге')
fig.tight_layout();fig.savefig('predefense_confusion.png',dpi=160);plt.show()

## 5. Решение о готовности (56–75 мин)

Технические `assert` подтверждают лишь узкую часть требований. Сделайте отдельный аудит готовности к внедрению: данные, мониторинг, человек в контуре, производительность, отказоустойчивость, дрейф, безопасность и ответственность. Для каждого пункта укажите **доказательство**, статус и действие. Пустое доказательство = статус не пройден, даже если модель показала хороший AP.

In [ ]:
OUT=Path('results_22');OUT.mkdir(exist_ok=True)
items=[
 ('Данные','Происхождение, права и разрешение на использование','нет документов'),
 ('Данные','Репрезентативность реальных заказов и периодов','учебная синтетика/нет независимого подтверждения'),
 ('ML','Нет утечки будущих признаков','проверен список FEATURES, нужен экспертный аудит'),
 ('ML','Отдельный окончательный тестовый период','отсутствует подтверждение независимости'),
 ('ML','Калибровка вероятностей и цена ошибок подтверждены','не подтверждено на реальных данных'),
 ('Артефакты','Схема, версии, препроцессор и веса восстановимы','проверка числового восстановления PASS'),
 ('Инфраструктура','SLA p95 задержки и нагрузочное тестирование','не выполнялось'),
 ('Эксплуатация','Мониторинг дрейфа, падений и обратная связь','не реализовано'),
 ('Эксплуатация','План rollback, резерв и контроль версий','не реализовано'),
 ('Ответственность','Человек подтверждает значимые действия','требуется регламент'),
 ('Безопасность','Контроль ПД и доступа, аудит событий','требуется анализ'),
 ('Качество','Разбор ошибок по группам и сезону','требуется проверка')]
readiness=pd.DataFrame([{'раздел':g,'критерий':q,'статус':'НЕ ПОДТВЕРЖДЕНО','доказательство/ограничение':why,'действие':'[назначить владельца и срок]'} for g,q,why in items])
readiness.to_csv(OUT/'deployment_readiness.csv',index=False,encoding='utf-8-sig')
display(readiness[['раздел','критерий','статус']])
print('Учебный вердикт по умолчанию: «техническая предзащита возможна; внедрение не разрешено».')

## 6. Карточка модели и пяти-минутная предзащита (75–90 мин)

**Структура доклада:** (0:00–0:45) задача и момент прогноза; (0:45–1:30) данные и split; (1:30–2:30) baseline vs MLP и статистика seed; (2:30–3:30) потери/стоимость/ограничения; (3:30–4:30) восстановление инференса; (4:30–5:00) решение комиссии и план.

**Вопросы комиссии:** почему сеть, а не логистическая регрессия; что запрещено в `X`; почему `test` не должен участвовать в выборе; почему AP не равен F1; как обнаружить дрейф; кто принимает решение по сигналу; что потребуется до запуска на предприятии.

In [ ]:
exp=pd.read_csv(R/'experiments.csv')
model_card=f"""# Model Card — учебный проект

## Назначение и граница прогноза
Прогноз задержки заказа более чем на 15 минут при создании заказа. Единица — заказ.

## Источник данных
{manifest['origin']}
Размер после аудита: {manifest['work_rows']} заказов. SHA-256 источника: {manifest.get('data_sha256')}.

## Признаки и утечки
Разрешены: {', '.join(FEATURES)}. Запрещены: {', '.join(manifest['forbidden_predictors'])}.

## Протокол
Хронологическое 60/20/20. Train={manifest['splits']['train']}, validation={manifest['splits']['validation']}, test={manifest['splits']['test']}. Идентификаторы не пересекаются.

## Модель и параметры
{config['model']}; width={config['width']}, dropout={config['dropout']}, threshold={tau}. Seed={config['seed']}.
Best epoch={int(exp.loc[(exp.model==manifest['selected_model']) & (exp.seed==config['seed']),'best_epoch'].iloc[0])}.

## Доказательства по validation (НЕ test)
AP={metrics['AP']:.4f}; F1={metrics['F1']:.4f}; Recall={metrics['Recall']:.4f}; Precision={metrics['Precision']:.4f}; Brier={metrics['Brier']:.4f}.
Условная стоимость на 100 заказов: {metrics['cost_100_rub']:.2f} ₽ (C_FP=60 ₽, C_FN=300 ₽; не тариф предприятия).

## Воспроизводимость
Повторное восстановление прогнозов: max abs diff = {maxerr:.2e}; веса и препроцессор сохранены отдельно.

## Ограничения и запрет на внедрение
Синтетические/учебные данные не подтверждают переносимость на реальные процессы; test ранее мог просматриваться в практиках. Независимая окончательная оценка неизвестна.
Не получены права на реальные данные, нагрузочные замеры, мониторинг дрейфа, план отката, оценка рисков ПД, доказательства экономической эффективности.

## Человеческий контроль и решение
Модель — рекомендательная; действия диспетчера не автоматизируются без регламента. Рекомендация: предзащита учебного прототипа, не промышленный запуск.

## Что необходимо сделать
1. Провести внешний тест на новом периоде.
2. Подтвердить причины и цены ошибок.
3. Провести испытания по группам и дрейфу.
4. Измерить p95 latency, построить мониторинг и rollback.
5. Утвердить ответственных и порядок обработки сигналов.
"""
(OUT/'model_card.md').write_text(model_card,encoding='utf-8')
(OUT/'predefense_speech.md').write_text("""# План защиты — 5 минут

1. Какой заказ прогнозируем и когда (45 секунд)
2. Почему именно эти признаки, данные и разбиение (45 секунд)
3. Сравнение baseline и MLP, seed, AP и цена ошибки (60 секунд)
4. Воспроизводимость, архитектура и checkpoint (60 секунд)
5. Риски, ограничения, Go / No-Go и план закрытия (90 секунд)

Покажите одну таблицу экспериментов, один график кривых обучения и один результат повторной загрузки модели.
""",encoding='utf-8')
print('Подготовлены:',[q.name for q in OUT.iterdir()])
print('Сохраните заполненную таблицу готовности и Model Card. Не подменяйте демонстрационные результаты реальными.')

### Что сдавать

`22_student.ipynb`, заполненную `deployment_readiness.csv`, `model_card.md`, `predefense_speech.md`, снимок/протокол демонстрации восстановленной модели и итоговый отчёт 3–5 страниц. К защите должны быть доступны артефакты 21, таблицы экспериментов и численно проверяемые выводы.  
**Критерий зачёта:** все ограничения честно отражены; неподтверждённые эксплуатационные критерии не отмечены как выполненные.